In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install diffusers transformers accelerate safetensors flask flask-cors


Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image
from io import BytesIO
import os

In [ ]:
!pip install -U diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 safetensors==0.4.3


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading base Stable Diffusion model...")
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(DEVICE)

print(" Model loaded successfully!")


Loading base Stable Diffusion model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


 Model loaded successfully!


In [ ]:
app = Flask(__name__)
CORS(app)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
print("Loading base Stable Diffusion model...")
base_model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    base_model_id, torch_dtype=torch.float16
)
pipe.to(DEVICE)


Loading base Stable Diffusion model...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


StableDiffusionPipeline {
  "_class_name": "StableDiffusionPipeline",
  "_diffusers_version": "0.30.0",
  "_name_or_path": "runwayml/stable-diffusion-v1-5",
  "feature_extractor": [
    "transformers",
    "CLIPImageProcessor"
  ],
  "image_encoder": [
    null,
    null
  ],
  "requires_safety_checker": true,
  "safety_checker": [
    "stable_diffusion",
    "StableDiffusionSafetyChecker"
  ],
  "scheduler": [
    "diffusers",
    "PNDMScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [ ]:
LORA_PATHS = {
    "anime": "/content/anime_lora.safetensors",
    "realistic": "/content/realistic_lora.safetensors"
}


In [ ]:
def apply_lora(pipe, lora_path, alpha=0.75):
    import safetensors.torch
    weights = safetensors.torch.load_file(lora_path)
    for key in weights:
        if key in pipe.unet.state_dict():
            pipe.unet.state_dict()[key] += alpha * weights[key]


In [ ]:
@app.route("/generate", methods=["POST"])
def generate_image():
    data = request.get_json()
    prompt = data.get("prompt", "")
    style = data.get("style", "realistic")

    if os.path.exists(LORA_PATHS[style]):
        apply_lora(pipe, LORA_PATHS[style])
        print(f"Applied {style} LoRA")

    with torch.autocast("cuda"):
        image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    buffer.seek(0)
    return send_file(buffer, mimetype="image/png")


In [ ]:
!pip install pyngrok
from pyngrok import ngrok
import os

# Put your ngrok auth token here (the long string you had earlier)
os.environ['NGROK_AUTH_TOKEN'] = '34mP4pTTHmzflKgiWPHNI3LANG4_2mJdyENQ8JCPaPHpcfwwB'

def start_server(port=5000):
    # Set auth token
    ngrok_token = os.environ.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(ngrok_token)

    # Start ngrok tunnel
    public_url = ngrok.connect(port).public_url

    print("\n" + "="*60)
    print("🚀 Colab Backend Started!")
    print("="*60)
    print(f"\n🔗 COPY THIS PUBLIC URL: {public_url}")
    print("\n" + "="*60 + "\n")

    # Run Flask app
    app.run(host='0.0.0.0', port=port)

if __name__ == "__main__":
    start_server(port=5000)



🚀 Colab Backend Started!

🔗 COPY THIS PUBLIC URL: https://a66e-34-16-244-247.ngrok-free.app


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:werkzeug:127.0.0.1 - - [22/Apr/2026 17:31:12] "POST /generate HTTP/1.1" 200 -


  0%|          | 0/30 [00:00<?, ?it/s]

...

INFO:werkzeug:127.0.0.1 - - [22/Apr/2026 17:39:34] "POST /generate HTTP/1.1" 200 -


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:werkzeug:127.0.0.1 - - [22/Apr/2026 17:40:05] "POST /generate HTTP/1.1" 200 -


  0%|          | 0/30 [00:00<?, ?it/s]

INFO:werkzeug:127.0.0.1 - - [22/Apr/2026 17:40:59] "POST /generate HTTP/1.1" 200 -
/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:140: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", 